In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pendulibrary.integrate import integrate_state
from pendulibrary.continuation import adaptive_cont
from pendulibrary.targeter import dc_underconstrained
from pendulibrary.common_targetters import single_fixed
from pendulibrary.interpolate import interp_hermite
from scipy.interpolate import CubicSpline

int_tol = 1e-14

# Read in the file

In [ ]:
fname = "DDsp"

data = np.load(f"../database/{fname}.npz")
x0s_in = data["x0s"]
periods_in = data["periods"]
tangents_in = data["tangents"]
eigs_in = data["eigs"]
Lr, Mr = data["params"]

ind_fix = np.argmin(np.std(x0s_in), axis=0)
Xs_init = np.column_stack((x0s_in, periods_in))
Xs_init = np.delete(Xs_init, ind_fix, 1)

# Refine the grid at candidate bifurcation points

In [ ]:
targ = single_fixed(ind_fix, x0s_in[ind_fix][0], 2, Lr, Mr, int_tol)
func = targ.g_dg_stm
f_func = lambda nu: [nu - 2, nu + 2, nu + 1, nu]

tangents = tangents_in.copy()
eigs = eigs_in.copy()
Xs = Xs_init.copy()

for jj in tqdm(range(5)):
    nus = np.sum(eigs, axis=1).real - 2
    Ss = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs, axis=0), axis=1)))
    splines = [CubicSpline(Ss, f) for f in f_func(nus)]
    allroots = np.sort(
        np.concat([spline.derivative().roots(extrapolate=False) for spline in splines])
    )
    allroots = np.unique(allroots[allroots < Ss[-1] / 2])
    allroots = allroots[
        [np.any([np.abs(sp(rt)) < 1e-2 for sp in splines]) for rt in allroots]
    ]
    insert_loc = np.searchsorted(Ss, allroots)

    Xs_guess = interp_hermite(Ss, Xs.T, tangents.T, allroots)[1].T

    Xnew = []
    eignew = []
    tannew = []
    for Xg in Xs_guess:
        X, DF, stm = dc_underconstrained(Xg, func, 1e-12, max_iter=25)
        Xnew.append(X.copy())
        eignew.append(np.linalg.eigvals(stm).copy())
        tannew.append(np.linalg.svd(DF).Vh[-1].copy())
    Xs = np.insert(Xs, insert_loc, Xnew, axis=0)
    tangents = np.insert(tangents, insert_loc, tannew, axis=0)
    eigs = np.insert(eigs, insert_loc, eignew, axis=0)

for jj in tqdm(range(5)):
    nus = np.sum(eigs, axis=1).real - 2
    Ss = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs, axis=0), axis=1)))
    splines = [CubicSpline(Ss, f) for f in f_func(nus)]
    allroots = np.sort(
        np.concat([spline.roots(extrapolate=False) for spline in splines])
    )
    insert_loc = np.searchsorted(Ss, allroots)

    Xs_guess = interp_hermite(Ss, Xs.T, tangents.T, allroots)[1].T

    Xnew = []
    eignew = []
    tannew = []
    for Xg in Xs_guess:
        X, DF, stm = dc_underconstrained(Xg, func, 1e-12, max_iter=25)
        Xnew.append(X.copy())
        eignew.append(np.linalg.eigvals(stm).copy())
        tannew.append(np.linalg.svd(DF).Vh[-1].copy())
    Xs = np.insert(Xs, insert_loc, Xnew, axis=0)
    tangents = np.insert(tangents, insert_loc, tannew, axis=0)
    eigs = np.insert(eigs, insert_loc, eignew, axis=0)
nus = np.sum(eigs, axis=1).real - 2
Ss = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs, axis=0), axis=1)))

# nu_init = (np.sum(eigs, axis=1) - 2).real
# Ss_init = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs, axis=0), axis=1)))
# f1 = nu_init - 2
# f2 = nu_init + 2
# f3 = nu_init + 1
# f4 = nu_init
# root_fs = [f1, f2, f3, f4]

In [ ]:
allroots

# Plot the new suspected bifurcation points

In [ ]:
splines = [CubicSpline(Ss, f) for f in f_func(nus)]
allroots_1 = [spline.roots(extrapolate=False) for spline in splines]
allroots_2a = np.sort(
    np.concat([spline.derivative().roots(extrapolate=False) for spline in splines])
)
allroots_2a = np.unique(allroots_2a[allroots_2a < Ss[-1] / 2])
allroots_2 = [allroots_2a[np.abs(sp(allroots_2a)) < 1e-3] for sp in splines]
allroots0 = [np.sort(np.concat((ar1, ar2))) for ar1, ar2 in zip(allroots_1, allroots_2)]
allroots = [[], [], [], []]
for ii in range(4):
    allroots[ii].append(allroots0[ii][0])
    for rt in allroots0[ii][1:]:
        if np.abs(rt - allroots[ii][-1]) > 1e-3:
            allroots[ii].append(rt)
    allroots[ii] = np.array(allroots[ii])

cols = ["m", "g", "b", "c"]
labels = ["Bifurcation", "Double", "Triple", "Quadrouple"]

fig, ax = plt.subplots()
plt.plot([Ss[0], Ss[-1]], [0, 0], "--k")
for ii in range(4):
    ax.plot(Ss, f_func(nus)[ii], f"-{cols[ii]}", label=labels[ii])
    for jj in range(len(allroots[ii])):
        ax.plot(
            allroots[ii][jj],
            0,
            ".",
            c=cols[ii],
            label=f"P{ii+1}:{jj} @ s={allroots[ii][jj]:.4f}",
            ms=10,
        )
ax.set(ylim=(-3, 3))
fig.legend(bbox_to_anchor=(1.0, 0.5), loc="center left")
fig.tight_layout()
plt.show()

# Find the exact new bifurcation points

In [ ]:
tangs_bif = [[], [], [], []]
tangs_2_bif = [[], [], [], []]
Xs_bif = [[], [], [], []]
eigs_bif = [[], [], [], []]
Xs_guess = [interp_hermite(Ss, Xs.T, tangents.T, rts)[1].T for rts in allroots]
for ii in range(4):
    for jj in range(len(Xs_guess[ii])):
        X, df, stm = dc_underconstrained(Xs_guess[ii][jj], func, 1e-13, max_iter=25)
        X[-1] *= 1 + ii
        X, df, stm = dc_underconstrained(X, func, 5e-13, max_iter=30)
        Xs_bif[ii].append(X.copy())
        tangs_bif[ii].append(np.linalg.svd(DF).Vh[-2])
        tangs_2_bif[ii].append(np.linalg.svd(DF).Vh[-3])
        print(np.linalg.svd(df).S)
        eigs_bif[ii].append(np.linalg.eigvals(stm))

# Continue a whole family of em

In [ ]:
cont_kwargs = dict(
    s0=6e-3, s_min=1e-5, S=12.5, tol=1e-10, max_iter=15, target_iter=6, rate=1.15
)


bif_type = 2
j_bif = 0
Xs_post, eig_vals_post, (DFs_post, tangents_post, stms_post) = adaptive_cont(
    Xs_bif[bif_type][j_bif],
    func,
    tangs_bif[bif_type][j_bif],
    **cont_kwargs,
    exact_tangent=True,
)

In [ ]:
x0s_new = np.array([targ.get_x0(X) for X in Xs_post])
periods_new = np.array([targ.get_period(X) for X in Xs_post])
np.savez(
    "../database/DDsp-P4b-true",
    x0s=x0s_new,
    periods=periods_new,
    eigs=eig_vals_post,
    tangents=tangents_post,
    params=np.array([Lr, Mr]),
)

# Optional plotters

In [ ]:
from pendulibrary.plotters import plot_timeline_grid, make_gif

Ss_new = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs_post, axis=0), axis=1)))

N = 6
for j in range(N):
    s = j * Ss_new[-1] / N
    iii = np.argmin(np.abs(Ss_new - s))
    ts, xs, fs = integrate_state(x0s_new[iii], periods_new[iii], Lr, Mr, 1e-14)
    fig = plot_timeline_grid(xs, ts, fs, Lr)
    fig.savefig(f"fig_{j}.png")
    plt.close(fig)

In [ ]:
ts, xs, fs = integrate_state(x0s_new[0], periods_new[0], Lr, Mr, 1e-14)
# make_gif(xs, ts, fs, Lr, "test3", 150, (4, 4))

In [ ]:
plt.plot(xs.T)

In [ ]:
plt.plot(xs.T)